In [ ]:
%load_ext autoreload
%autoreload 2

import sys, pathlib
p = pathlib.Path.cwd()
for q in (p, *p.parents):
    s = q / "src" / "ftbp"   # <- change "ftbp" if you rename the package
    if s.exists():
        sys.path.insert(0, str(s.parent))  # add .../src
        break
else:
    raise RuntimeError("src/ftbp not found")

In [ ]:
import numpy as np
import pandas as pd
import itertools
from math import comb
from scipy.optimize import brentq
from scipy.stats import norm, cauchy, uniform
import scipy.stats
from statsmodels.robust.scale import hubers_scale
from ftbp.two_stage import *

In [ ]:
ms = [10 + i * 10 for i in range(15)]
print(ms)
etas = [0.1 + i * 0.1 for i in range(10)]
print(etas)

In [ ]:
### the gap experiment
from tqdm import tqdm
n = 1000
# ns = [100]
ms = [10 + i * 10 for i in range(15)]
# ms = [460]
M = 100
seeds = [53 + i for i in range(M)]
dists = [norm, cauchy, uniform]
# dists = [uniform]
loss_type = 'huber'  # or 'logcosh'
if loss_type == 'huber':
    delta = 1.345
elif loss_type == 'logcosh':
    delta = 1.2047
elif loss_type == 'concordant':
    delta = 1.479

all_results = []
for m in ms:
    for dist in dists:
        for seed in seeds:
            x = dist.rvs(size=n, random_state=seed)
            if dist is uniform:
                x = x - 0.5 # centered
            if dist is uniform:
                distribution = 'uniform'
            elif dist is norm:
                distribution = 'normal'
            elif dist is cauchy:
                distribution = 'Cauchy'
            eta_lower = max(two_stage_eta_lower_minus(x, m, delta, loss_type, 'huber', distribution), two_stage_eta_lower_plus(x, m, delta, loss_type, 'huber', distribution))
            eta_upper = max(two_stage_eta_upper_minus(x, m, delta, loss_type, 'huber', distribution), two_stage_eta_upper_plus(x, m, delta, loss_type, 'huber', distribution))

            all_results.append({
                'm': m,
                'dist': getattr(dist, 'name', dist.__class__.__name__),
                'seed': seed,
                'eta_lower': eta_lower,
                'eta_upper': eta_upper
            })


df = pd.DataFrame(all_results)

In [ ]:
df

In [ ]:
# check if eta_upper > eta_lower for all rows in df, in numbers
# if not find the one that is not greater
for index, row in df.iterrows():
    if row['eta_upper'] <= row['eta_lower']:
        print(f"Row {index} is not greater: {row['eta_upper']} <= {row['eta_lower']}")


In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# assume df_avg already exists from previous steps:
#   columns=['n','Effect Size','Lower Bound of Power','Upper Bound of Power']
sns.set_style("whitegrid", {'grid.color': '#F2F2F2'})  # e.g. “darkgrid”, “ticks”, “white”
sns.set_context("talk")
palette = sns.color_palette("husl", 3)

# 1. Melt to long form
df_avg = (
    df
    .groupby(['m', 'dist'], as_index=False)
    .agg({'eta_lower': 'mean', 'eta_upper': 'mean'})
)

marker_map = {'norm': 'o', 'cauchy': 's', 'uniform': '^'}

# 3. Single‐plot
plt.figure(figsize=(8, 6))
ax = sns.lineplot(
    data=df_avg,
    x='m', y='eta_lower',
    hue='dist',                 # color by bound type
    style='dist',            # line style by dist
    markers=marker_map,
    palette=palette,
    legend='full',
    hue_order=['norm', 'cauchy', 'uniform'],
    alpha=0.8,
)

sns.lineplot(
    data=df_avg,
    x='m', y='eta_upper',
    hue='dist',                 # color by bound type
    style='dist',            # line style by dist
    markers=marker_map,
    palette=palette,
    legend=False,
    hue_order=['norm', 'cauchy', 'uniform'],
    alpha=0.8,
)

i = -1
for dist in ['norm', 'cauchy', 'uniform']:
    i += 1
    sub = df_avg[df_avg['dist'] == dist]
    ax.fill_between(sub['m'],
                    sub['eta_lower'],
                    sub['eta_upper'],
                    alpha=0.1, color=palette[i])

plt.xlabel(r'$m$')
plt.ylabel(r'$\eta_{m/n}$')
handles, labels = ax.get_legend_handles_labels()
new_labels = ["Normal", "Cauchy", "Uniform"]  # as many as you have
ax.legend(loc='upper center', bbox_to_anchor=(0.478, -0.15), ncols=7, frameon=False, handles=handles, labels=new_labels)
plt.tight_layout()
# plt.show()
plt.savefig('eta_two_stage.pdf', bbox_inches='tight')


In [ ]:
### the gap experiment
from tqdm import tqdm
n = 1000
# ns = [100]
ms = [10 + i * 10 for i in range(15)]
etas = [0.1 + i * 0.1 for i in range(10)]
# ms = [460]
M = 100
seeds = [53 + i for i in range(M)]
dists = [norm, cauchy, uniform]
loss_type = 'huber'  # or 'logcosh'
if loss_type == 'huber':
    delta = 1.345
elif loss_type == 'logcosh':
    delta = 1.2047
elif loss_type == 'concordant':
    delta = 1.479

all_results = []
for eta in etas:
    for dist in dists:
        for seed in tqdm(seeds):
            x = dist.rvs(size=n, random_state=seed)
            if dist is uniform:
                x = x - 0.5 # centered
            if dist is uniform:
                distribution = 'uniform'
            elif dist is norm:
                distribution = 'normal'
            elif dist is cauchy:
                distribution = 'Cauchy'
            BP_lower = max(two_stage_BP_lower_minus(x, eta, delta, loss_type, 'huber', distribution), two_stage_BP_lower_plus(x, eta, delta, loss_type, 'huber', distribution))
            BP_upper = max(two_stage_BP_upper_minus(x, eta, delta, loss_type, 'huber', distribution), two_stage_BP_upper_plus(x, eta, delta, loss_type, 'huber', distribution))

            all_results.append({
                'dist': getattr(dist, 'name', dist.__class__.__name__),
                'seed': seed,
                'BP_lower': BP_lower,
                'BP_upper': BP_upper,
                'eta': eta
            })

df = pd.DataFrame(all_results)

In [ ]:
df[df['eta'] == 0.1]

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# assume df_avg already exists from previous steps:
#   columns=['n','Effect Size','Lower Bound of Power','Upper Bound of Power']
sns.set_style("whitegrid", {'grid.color': '#F2F2F2'})  # e.g. “darkgrid”, “ticks”, “white”
sns.set_context("talk")
palette = sns.color_palette("husl", 3)

# 1. Melt to long form
df_avg = (
    df
    .groupby(['eta', 'dist'], as_index=False)
    .agg({'BP_lower': 'mean', 'BP_upper': 'mean'})
)

df_avg['BP_lower'] = df_avg['BP_lower'] / 1000  # scale down for better visualization
df_avg['BP_upper'] = df_avg['BP_upper'] / 1000  #

marker_map = {'norm': 'o', 'cauchy': 's', 'uniform': '^'}

# 3. Single‐plot
plt.figure(figsize=(8, 6))
ax = sns.lineplot(
    data=df_avg,
    x='eta', y='BP_lower',
    hue='dist',                 # color by bound type
    style='dist',            # line style by dist
    markers=marker_map,
    palette=palette,
    legend='full',
    hue_order=['norm', 'cauchy', 'uniform'],
    alpha=0.8,
)

sns.lineplot(
    data=df_avg,
    x='eta', y='BP_upper',
    hue='dist',                 # color by bound type
    style='dist',            # line style by dist
    markers=marker_map,
    palette=palette,
    legend=False,
    hue_order=['norm', 'cauchy', 'uniform'],
    alpha=0.8,
)

i = -1
for dist in ['norm', 'cauchy', 'uniform']:
    i += 1
    sub = df_avg[df_avg['dist'] == dist]
    ax.fill_between(sub['eta'],
                    sub['BP_lower'],
                    sub['BP_upper'],
                    alpha=0.1, color=palette[i])

plt.xlabel(r'$\eta$')
plt.ylabel(r'$BP_\eta$')
handles, labels = ax.get_legend_handles_labels()
new_labels = ["Normal", "Cauchy", "Uniform"]  # as many as you have
ax.legend(loc='upper center', bbox_to_anchor=(0.478, -0.15), ncols=7, frameon=False, handles=handles, labels=new_labels)
plt.tight_layout()
# plt.show()
plt.savefig('bp_two_stage.pdf', bbox_inches='tight')
